### Ejercicio 1

No, Bash no es una buena opción para desarrollar una aplicación web. Aunque Lucas y Martín propongan usarlo porque es una herramienta que conocen, Tomás tiene razón en dudar.

Bash es un lenguaje de scripting diseñado principalmente para automatizar tareas del sistema operativo, manipular archivos o gestionar servidores. No está pensado para manejar solicitudes HTTP, conectar con bases de datos relacionales ni renderizar interfaces en el navegador.

Para construir la aplicación web que plantean (y además escalarla con funcionalidades como estadísticas o registro de lesiones), deberían optar por tecnologías adecuadas como JavaScript/Node.js, Python, Java o PHP.

### Ejercicio 2

GitHub es una plataforma en la nube que utiliza Git, un sistema de control de versiones.

Beneficia al equipo porque permite:

- Trabajar en el mismo código sin sobrescribir el trabajo de otros.
- Mantener un historial de cambios (y volver a versiones anteriores si algo falla).
- Probar nuevas funcionalidades en ramas (branches) sin afectar el proyecto principal.


### Ejercicio 3

Modelo de base de datos (SQL)

Al ser un grupo de amigos, considero que normalmente no hay equipos fijos.  
Por lo tanto, los jugadores van a ir mezclandose.  

```sql
-- jugadores (id, nombre_completo, posicion_preferida, fecha_nacimiento, nacionalidad, dni)
CREATE TABLE jugadores(
    id SERIAL PRIMARY KEY,
    nombre_completo VARCHAR(100) NOT NULL,
    posicion_preferida VARCHAR(50),
    fecha_nacimiento DATE,
    nacionalidad VARCHAR(50) DEFAULT 'Argentina',
    dni INT,
    email VARCHAR(255) UNIQUE NOT NULL,
    contrasenia VARCHAR(255) NOT NULL
);

-- partidos (id, fecha_hora, lugar, goles_local, goles_visitante)
CREATE TABLE partidos(
    id SERIAL PRIMARY KEY,
    fecha_hora TIMESTAMP NOT NULL,
    lugar TEXT NOT NULL,
    goles_local SMALLINT,
    goles_visitante SMALLINT,
    inscripcion_desde TIMESTAMP NOT NULL,
    inscripcion_hasta TIMESTAMP NOT NULL
);
-- En el caso de goles_* podría ser un tipo de dato que solo contemple número >= 0

-- jugadores_partidos
CREATE TABLE jugadores_partidos(
    id_jugador INT REFERENCES(jugadores),
    id_partido INT REFERENCES(partidos),
    es_local BOOL NOT NULL,
    goles_anotados SMALLINT NOT NULL DEFAULT 0,
    asistencias_hechas SMALLINT NOT NULL DEFAULT 0,
    PRIMARY KEY (id_jugador, id_partido)
);

-- inscripciones (partido, jugador, fecha_inscripcion)
CREATE TABLE inscripciones(
    id_jugador INT REFERENCES(jugadores),
    id_partido INT REFERENCES(partidos),
    fecha_inscripcion TIMESTAMP NOT NULL,
    PRIMARY KEY (id_jugador, id_partido)
);
```

Explicaciones cubriendo los 3 items del enunciado:  
  
Tendríamos una tabla para los jugadores, la cual contendría campos como un ID autoincremental, el nombre del jugador, su posicion preferida, su fecha de nacimiento, su email y contraseña para iniciar sesión en el sistema. La idea es que cada persona del grupo se registre y complete sus datos.  

También tendríamos una tabla para los partidos, con los campos de la fecha y hora del mismo, la ubicación, también un ID autoincremental, la cantidad de goles del equipo local y la cantidad de goles del equipo visitante.  

Otra tabla es la de inscripciones, la cual contiene una FK hacia un jugador y otra hacia un partido, junto con la fecha de inscripción del jugador. Esta tabla nos sirve para registrar que un jugador se anotó para jugar un partido en particular. La PK es la tupla (id_jugador, id_partido).  

Por último, una tabla jugadores_partidos que nos servirá para indicar qué jugadores jugaron qué partidos. Esta tabla representa una relación mucho a muchos. Tendrá las mismas FK que la tabla anterior, un campo booleano "es_local", indicando si el jugador jugó para el equipo local, y los goles anotados y asistencias hechas por el jugador en ese partido.  

Las entidades principales serían los jugadores y los partidos.  

Las relaciones que tenemos son desde jugadores_partidos e inscripciones hacia la tabla de jugadores y la tabla de partidos. Las FK sirven para referencias la PK de otra tabla, en este caso la de jugadores.  

### Ejercicio 4

Goles hechos por partido (fecha) por cada jugador (nombre completo).

```sql
SELECT partidos.fecha_hora, jugadores.nombre_completo, jp.goles_anotados
FROM jugadores_partidos jp
JOIN jugadores ON jp.id_jugador = jugadores.id
JOIN partidos ON jp.id_partido = partidos.id
WHERE jp.goles_anotados != 0
-- `<>` es lo mismo que `!=`
-- En ese where se sacan los jugadores que no anotaron goles
ORDER BY 
    partidos.fecha_hora, 
    jp.goles_anotados DESC, 
    jp.asistencias_hechas DESC, 
    jugadores.nombre_completo;
```

Se muestra, para cada partido, su fecha y la cantidad de goles anotados por cada jugador (con su nombre), ordenada primero por fecha del partido (de más viejo a más nuevo) y luego por cantidad de goles hechos (de mayor a menor). En caso de empate de goles, se define por cantidad de asistencias. Si aún hubiera empate, se define por orden alfabético del nombre del jugador. Solo se muestran los jugadores que marcaron al menos un gol en cada partido.



** Forma alternativa con multiple FROM (Mala práctica por posibles problemas de prod. cartesiano)

```sql
SELECT p.fecha_hora, j.nombre_completo, jp.goles_anotados
FROM jugadores_partidos jp, jugadores j, partidos p
WHERE jp.id_jugador = j.id AND jp.id_partido = p.id AND jp.goles_anotados != 0
ORDER BY p.fecha_hora, jp.goles_anotados DESC, jp.asistencias_hechas DESC, j.nombre_completo;
```


### Ejercicio 5

El equipo se encuentra principalmente en la etapa de Análisis de Requerimientos.

Están identificando:

el problema (desorganización)
la solución (aplicación web)

También están comenzando la etapa de diseño al discutir:

tecnologías
arquitectura (web vs móvil)

### Ejercicio 6

Docker permite crear entornos de ejecución iguales para todos los desarrolladores.

Esto evita problemas como:

“en mi máquina funciona, pero en la tuya no”

Permite:

empaquetar aplicación + dependencias
facilitar despliegue
asegurar consistencia entre entornos


### Extra: Ejercicio 7
Consigna:  
Tomas recolectó un montón de equipos que podrían usar el sistema y los guardó en un archivo csv.
El formato del archivo es: 
NombreEquipo;PartidosPorSemana;TeléfonoContacto; MailContacto

Quieren arrancar contactando a quienes jueguen 1 vez por semana y el Mail sea un gmail.com.

Escribir una expresión regular que liste los equipos a contactar

Solución:
```regex
^[^;]+;1;[^;]+;[^@]+@gmail\.com$
```

- `^`: inicio de línea
- `[^;]+`: nombre del equipo
- `;1;`: juega 1 vez por semana
- `[^;]+`: teléfono
- `[^@]+`: usuario del mail
- `gmail\.com`: dominio gmail
- `$`: fin de línea
